# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 7 — Gold: agregações e KPIs

## 🎯 Objetivo

Construir a tabela Gold que o BI vai consumir amanhã — uma linha por segmento, com os KPIs prontos.

**Rota B - DuckDB + Python/Google Colab**

## Configuracao Inicial


In [1]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio
import os
import shutil
import duckdb

os.makedirs("bigdata/silver", exist_ok=True)
os.makedirs("bigdata/gold", exist_ok=True)

# Abre uma conexao DuckDB
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Carregar a Silver do Lab 6

In [2]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
# (fraud_labels.csv nao e necessario para este lab - e material de outro exercicio)
#from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [3]:
# Copia os CSVs para a estrutura Raw do projeto
import shutil, os

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)

for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [4]:
# Reconstroi a Bronze com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [5]:
# Reconstroi a Silver com as mesmas colunas derivadas do Lab 6.
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



## Passo 1 — Gold principal por segmento

In [6]:
# Cria a Gold agregando a Silver por segmento (Premium, Standard, High-Risk)
# valor_em_risco soma o amount apenas das transacoes marcadas como fraude
con.sql("""
CREATE OR REPLACE TABLE gold_fraud_risk AS
SELECT
  segment,
  COUNT(*) AS total_transacoes,
  SUM(amount) AS valor_total,
  ROUND(AVG(amount), 2) AS ticket_medio,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS qtd_fraudes,
  ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct,
  SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS valor_em_risco
FROM silver_transactions
GROUP BY segment
""")

# Confere o resultado ordenado pela taxa de fraude - High-Risk deve aparecer no topo
con.sql("SELECT * FROM gold_fraud_risk ORDER BY taxa_fraude_pct DESC").show()

┌───────────┬──────────────────┬────────────────────┬──────────────┬─────────────┬─────────────────┬────────────────────┐
│  segment  │ total_transacoes │    valor_total     │ ticket_medio │ qtd_fraudes │ taxa_fraude_pct │   valor_em_risco   │
│  varchar  │      int64       │       double       │    double    │   int128    │     double      │       double       │
├───────────┼──────────────────┼────────────────────┼──────────────┼─────────────┼─────────────────┼────────────────────┤
│ High-Risk │             9155 │ 1664640.2447309494 │       181.83 │         705 │             7.7 │ 126451.06910800934 │
│ Standard  │            29689 │  5479399.846734047 │       184.56 │         655 │            2.21 │ 106828.41910934448 │
│ Premium   │            61156 │ 11235041.576210976 │       183.71 │         473 │            0.77 │  81308.43799591064 │
└───────────┴──────────────────┴────────────────────┴──────────────┴─────────────┴─────────────────┴────────────────────┘



## Passo 2 — Gold de métricas diárias

In [7]:
# Cria a segunda Gold: volume de transacoes e fraudes por dia
# Usa year/month/day, que ja vieram derivados de ts na Silver (Lab 6)
con.sql("""
CREATE OR REPLACE TABLE gold_daily_metrics AS
SELECT year, month, day,
  COUNT(*) AS transacoes,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes
FROM silver_transactions
GROUP BY year, month, day
""")

# Verifica os 5 dias com mais fraudes - esperado concentrar perto do fim do mes
con.sql("""
SELECT year, month, day, fraudes
FROM gold_daily_metrics
ORDER BY fraudes DESC
LIMIT 5
""").show()

┌───────┬───────┬───────┬─────────┐
│ year  │ month │  day  │ fraudes │
│ int64 │ int64 │ int64 │ int128  │
├───────┼───────┼───────┼─────────┤
│  2023 │     4 │    27 │      16 │
│  2023 │     5 │    21 │      13 │
│  2023 │    12 │    22 │      13 │
│  2024 │     5 │    23 │      13 │
│  2024 │     5 │    25 │      12 │
└───────┴───────┴───────┴─────────┘



## Validacoes rapidas

In [8]:
# gold_fraud_risk deve ter exatamente 3 linhas, uma por segmento
con.sql("SELECT COUNT(*) AS n_segmentos FROM gold_fraud_risk").show()

# gold_daily_metrics deve ter em torno de 730 linhas (2 anos de dados diarios)
con.sql("SELECT COUNT(*) AS n_dias FROM gold_daily_metrics").show()

# valor_em_risco nao pode ficar zerado em nenhum segmento -
# se ficar, e sinal de que is_fraud nao esta sendo tratado como boolean
con.sql("SELECT segment, valor_em_risco FROM gold_fraud_risk WHERE valor_em_risco = 0").show()

┌─────────────┐
│ n_segmentos │
│    int64    │
├─────────────┤
│           3 │
└─────────────┘

┌────────┐
│ n_dias │
│ int64  │
├────────┤
│    672 │
└────────┘

┌─────────┬────────────────┐
│ segment │ valor_em_risco │
│ varchar │     double     │
└─────────┴────────────────┘
           0 rows         



## Passo 3 — Persistir as duas Golds (para o Lab 9 de export)

In [9]:
# Salva as duas tabelas Gold em Parquet
con.sql("COPY gold_fraud_risk TO 'bigdata/gold/fraud_risk.parquet' (FORMAT PARQUET)")
con.sql("COPY gold_daily_metrics TO 'bigdata/gold/daily_metrics.parquet' (FORMAT PARQUET)")

# Versao CSV da Gold de segmento tambem, para abrir direto em Excel/Metabase
con.sql("COPY gold_fraud_risk TO 'bigdata/gold/fraud_risk.csv' (HEADER, DELIMITER ',')")

print("Gold layer salva em bigdata/gold/")

Gold layer salva em bigdata/gold/


In [10]:
# # Baixa os arquivos da Gold para o computador local
# # (repita o download antes de encerrar a sessao, ja que bigdata/ e efemero no Colab)
# from google.colab import files as colab_files

# colab_files.download("bigdata/gold/fraud_risk.parquet")
# colab_files.download("bigdata/gold/daily_metrics.parquet")
# colab_files.download("bigdata/gold/fraud_risk.csv")

## Checkpoint

- [ ] `gold_fraud_risk` com 3 linhas (uma por segmento)
- [ ] `gold_daily_metrics` com aproximadamente 730 linhas (uma por dia)
- [ ] High-Risk aparece com a maior `taxa_fraude_pct`
- [ ] Nenhum segmento com `valor_em_risco` igual a zero
- [ ] Arquivos Parquet + CSV salvos em `bigdata/gold/` e baixados para a maquina local

---

**Proximo lab:** `DIA2_LAB08_EDA.md` - as 5 perguntas de negocio.